# Homework 5 — Deploying Machine Learning Models

Frozen lead-scoring release, 2026 cohort.

Setup used for every cell:

```bash
cd cohorts/2026/homework/05-deployment
uv sync --locked
```

The artifact is checked before use:

```text
1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin
```


## Question 1 — environment check

`uv --version` (setup check, not graded):

In [ ]:
import subprocess

print(subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip())


uv 0.13.5 (x86_64-unknown-linux-gnu)


## Question 2 — locked dependency

Which Scikit-Learn version is part of the 2026 reference environment?

Read straight from the lockfile instead of guessing:

In [ ]:
import re
from pathlib import Path

lock = Path("uv.lock").read_text(encoding="utf-8")
block = re.search(r'name = "scikit-learn"\nversion = "([^"]+)"', lock)
print("scikit-learn =", block.group(1))


scikit-learn = 1.7.2


**Answer: `1.7.2`**

Confirmed by `model_metadata.json`, which records
`"scikit_learn": "1.7.2"` for the artifact.

## Question 3 — load the model

Load `pipeline.bin` with `pickle` and call `predict_proba` for the given lead.

Note: the artifact is **read-only** here — it is never re-dumped.

In [ ]:
import pickle

LEAD_Q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41,
}

with open("pipeline.bin", "rb") as f_in:
    pipeline = pickle.load(f_in)

probability_q3 = float(pipeline.predict_proba([LEAD_Q3])[0, 1])

print("conversion_probability =", probability_q3)
print("rounded                =", round(probability_q3, 3))


conversion_probability = 0.5329944186432507
rounded                = 0.533


**Answer: `0.533`**

The lead is complete, so `normalize_record()` is a no-op here — verified:

In [ ]:
from model import normalize_record

raw = pipeline.predict_proba([LEAD_Q3])[0, 1]
normalized = pipeline.predict_proba([normalize_record(LEAD_Q3)])[0, 1]

print("identical =", raw == normalized)


identical = True


## Question 4 — serve the model

```bash
uv run uvicorn predict:app --host 0.0.0.0 --port 9696
```

Then call `POST /predict` with the second lead:

In [ ]:
import requests

LEAD_Q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74,
}

URL = "http://localhost:9696"

response = requests.post(f"{URL}/predict", json=LEAD_Q4).json()
print(response)
print("rounded =", round(response["conversion_probability"], 3))


{'conversion_probability': 0.769799, 'conversion': True}
rounded = 0.77


**Answer: `0.770`**

## Question 5 — look at the container configuration

Which Python base-image tag is declared in the `Dockerfile`?

In [ ]:
from pathlib import Path

first_line = Path("Dockerfile").read_text(encoding="utf-8").splitlines()[0]
print(first_line)
print()
print("tag =", first_line.split()[1].split("@")[0])


FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3

tag = python:3.11.15-slim-bookworm


**Answer: `python:3.11.15-slim-bookworm`**

## Question 6 — run the container

```bash
docker build -t zoomcamp-model:2026-hw5 .
docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5
```

Same request as Question 4 against the container:

In [ ]:
import requests

URL = "http://localhost:9696"

health = requests.get(f"{URL}/health").json()
response = requests.post(f"{URL}/predict", json=LEAD_Q4).json()

print("health  =", health)
print("predict =", response)
print()
print("rounded  =", round(response["conversion_probability"], 3))
print("checksum matches model_metadata.json =",
      health["model_sha256"] == "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1")


health  = {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
predict = {'conversion_probability': 0.769799, 'conversion': True}

rounded  = 0.77
checksum matches model_metadata.json = True


**Answer: `0.770`**

## Summary

| Question | Answer |
| --- | --- |
| 1 | `uv 0.13.5` (setup check, not graded) |
| 2 | `1.7.2` |
| 3 | `0.533` |
| 4 | `0.770` |
| 5 | `python:3.11.15-slim-bookworm` |
| 6 | `0.770` |

Question 4 and Question 6 return the same reference inference, as the homework
states. The `/health` endpoint reports the artifact checksum
`1646bbdc…29f1`, matching `model_metadata.json`.
